In [1]:
import numpy as np
import pandas as pd
from lib.dataprep import Cleaning, Imputation, RoughFeatureReduction, FeatureSelector, Balancer
from lib.datamodeling import DataModeler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold

In [2]:
X = pd.read_csv("../data/secom.data", sep=" ", header=None, na_values="NaN")
y = pd.read_csv("../data/secom_labels.data", sep=" ", header=None, names=["label", "timestamp"])

X_train, X_test, y_train, y_test = train_test_split(X, y["label"], test_size=0.2, random_state=42, stratify=y["label"])
print("Original:", X.shape, "| Train shape:", X_train.shape, "| Test shape:", X_test.shape)

Original: (1567, 590) | Train shape: (1253, 590) | Test shape: (314, 590)


In [6]:
# ---------------------------------------------- Training Data Preprocessing ----------------------------------------------
X_train_cleaned, has_invalid_strings = Cleaning().flag_missing_values(X_train)
print(f"Missing values flagged. Total Missing Values: {X_train_cleaned.isnull().sum().sum()}. DataFrame shape: {X_train_cleaned.shape}\n")

X_train_cleaned = Cleaning().cap_outlier_using_3sigma_rule(X_train_cleaned)
print(f"Outliers have been capped using the 3-sigma rule. DataFrame shape: {X_train_cleaned.shape}\n")

missingness_threshold = 0.6
X_train_cleaned = RoughFeatureReduction().remove_features_by_missingness_threshold(X_train_cleaned, missingness_threshold)
print(f"Features with more than {missingness_threshold*100}% missing values have been removed. DataFrame shape: {X_train_cleaned.shape}\n")

X_train_cleaned = RoughFeatureReduction().remove_constant_features(X_train_cleaned)
print(f"Constant features have been removed. DataFrame shape: {X_train_cleaned.shape}\n")

scaler = Imputation().fit_scaler(X_train_cleaned)
X_train_scaled = Imputation().transform_scaler(X_train_cleaned, scaler)
imputer = Imputation().fit_imputer(X_train_scaled)
X_train_imputed = Imputation().transform_imputer(X_train_scaled, imputer)
print(f"Training data has been scaled and imputed. Total Missing Values: {X_train_imputed.isnull().sum().sum()}. DataFrame shape: {X_train_imputed.shape}\n")

Missing values flagged. Total Missing Values: 33868. DataFrame shape: (1253, 590)

Outliers have been capped using the 3-sigma rule. DataFrame shape: (1253, 590)

Features with more than 60.0% missing values have been removed. DataFrame shape: (1253, 566)

Constant features have been removed. DataFrame shape: (1253, 450)

Training data has been scaled and imputed. Total Missing Values: 0. DataFrame shape: (1253, 450)



In [ ]:
# ---------------------------------------------- Test Data Preprocessing (X_test_reduced) ----------------------------------------------
# Scale using training scaler
X_test = X_test[X_train_cleaned.columns]
X_test_scaled = scaler.transform(X_test)

# Impute using training imputer
X_test_imputed = imputer.transform(X_test_scaled)

# Convert back to DataFrame with original column names
X_test_reduced = pd.DataFrame(X_test_imputed, columns=X_test.columns)

In [ ]:
# Evaluation with test data reduced
y_pred = rf_baseline_model.predict(X_test_reduced)
DataModeler().evaluate_model(y_test, y_pred)